# ask

> Answer questions about a video course using LLM tool use over the yttoc cache

In [ ]:
#| default_exp ask

## Design

`yttoc-ask` is a read-only reasoning shell. It exposes 2 thin tools (`get_summaries`, `get_xscript_range`) to an LLM via the OpenAI Responses API. The LLM decides what to fetch. Python only formats output.

See `docs/superpowers/specs/2026-04-16-yttoc-ask-design.md` for full design rationale.

In [ ]:
#| export
import json, sys
from pathlib import Path

In [ ]:
#| export
SYSTEM_PROMPT = """You are answering questions about a video course.
Use the provided tools to look up information. Do not fabricate content.
Answer in the user's language. If data is unavailable, say so."""

TOOLS = [
    {
        "type": "function",
        "name": "get_summaries",
        "description": "Return the full summaries.json for a cached video. Contains: video metadata (id, title, url, channel, duration), all sections (path, title, start/end seconds, summary, keywords, evidence with timestamp), and a full-video summary. Returns {\"error\": \"...\"} if the video is not prepared.",
        "parameters": {
            "type": "object",
            "properties": {
                "video_id": {"type": "string", "description": "YouTube video ID"}
            },
            "required": ["video_id"],
        },
    },
    {
        "type": "function",
        "name": "get_xscript_range",
        "description": "Return raw parsed transcript segments within a time range [start, end). Each segment is {start: float_seconds, end: float_seconds, text: string}. Use start/end values from get_summaries sections. Returns {\"error\": \"...\"} if captions are not available.",
        "parameters": {
            "type": "object",
            "properties": {
                "video_id": {"type": "string", "description": "YouTube video ID"},
                "start": {"type": "number", "description": "Start time in seconds"},
                "end": {"type": "number", "description": "End time in seconds"},
            },
            "required": ["video_id", "start", "end"],
        },
    },
]

ANSWER_SCHEMA = {
    "type": "json_schema",
    "name": "course_answer",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "answer": {"type": "string", "description": "Synthesized answer to the question"},
            "citations": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "video_id": {"type": "string"},
                        "seconds": {"type": "integer"},
                    },
                    "required": ["video_id", "seconds"],
                    "additionalProperties": False,
                },
            },
        },
        "required": ["answer", "citations"],
        "additionalProperties": False,
    },
}

## format_citations

In [ ]:
#| export
from yttoc.summarize import get_summaries as _read_summaries
from yttoc.xscript import get_xscript_range as _read_xscript_range
from yttoc.core import fmt_duration

def _find_section(sections: list[dict], seconds: int) -> dict | None:
    "Find the section containing the given timestamp. Returns None if no match."
    for s in sections:
        if s['start'] <= seconds < s['end']:
            return s
    return None

def format_citations(citations: list[dict], # [{video_id, seconds}, ...]
                     root: Path = None # Cache root for summaries lookup
                    ) -> list[str]: # Formatted citation lines
    "Resolve [{video_id, seconds}] into display lines with YouTube deep links."
    from yttoc.fetch import _DEFAULT_ROOT
    root = root or _DEFAULT_ROOT
    lines = []
    for i, c in enumerate(citations, 1):
        vid, sec = c['video_id'], c['seconds']
        url = f'https://youtu.be/{vid}?t={sec}'
        ts = fmt_duration(sec)
        sums = _read_summaries(vid, root)
        if 'error' in sums:
            lines.append(f'  [{i}] {vid} @ {ts}\n      {url}')
            continue
        title = sums['video'].get('title', vid)
        section = _find_section(sums['sections'], sec)
        if section:
            lines.append(f'  [{i}] {title} \u00a7{section["path"]} "{section["title"]}" @ {ts}\n      {url}')
        else:
            lines.append(f'  [{i}] {title} @ {ts}\n      {url}')
    return lines

In [ ]:
# Test: format_citations resolves {video_id, seconds} to display string
from tempfile import TemporaryDirectory

with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID1'; v.mkdir()
    (v / 'summaries.json').write_text(json.dumps({
        'video': {'id': 'VID1', 'title': 'Test Video', 'channel': 'C',
                  'url': 'https://www.youtube.com/watch?v=VID1',
                  'duration': 600, 'upload_date': '20260101'},
        'sections': [
            {'path': '1', 'title': 'Intro', 'start': 0, 'end': 300,
             'summary': 's', 'keywords': [], 'evidence': {'text': '', 'at': 0}},
            {'path': '2', 'title': 'Main', 'start': 300, 'end': 600,
             'summary': 's', 'keywords': [], 'evidence': {'text': '', 'at': 300}},
        ],
        'full': {'summary': '', 'keywords': [], 'evidence': {'text': '', 'at': 0}},
    }))

    citations = [{'video_id': 'VID1', 'seconds': 350}]
    lines = format_citations(citations, root)
    assert len(lines) == 1
    assert 'Test Video' in lines[0]
    assert '§2' in lines[0]
    assert '"Main"' in lines[0]
    assert 'https://youtu.be/VID1?t=350' in lines[0]
    assert '5:50' in lines[0]  # 350 seconds = 5:50
print('ok')

In [ ]:
# Test: format_citations handles seconds not in any section
with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID2'; v.mkdir()
    (v / 'summaries.json').write_text(json.dumps({
        'video': {'id': 'VID2', 'title': 'V2', 'channel': 'C',
                  'url': 'https://www.youtube.com/watch?v=VID2',
                  'duration': 100, 'upload_date': '20260101'},
        'sections': [
            {'path': '1', 'title': 'Only', 'start': 10, 'end': 50,
             'summary': 's', 'keywords': [], 'evidence': {'text': '', 'at': 10}},
        ],
        'full': {'summary': '', 'keywords': [], 'evidence': {'text': '', 'at': 0}},
    }))

    lines = format_citations([{'video_id': 'VID2', 'seconds': 80}], root)
    assert len(lines) == 1
    assert 'https://youtu.be/VID2?t=80' in lines[0]
print('ok')

## Tool dispatcher and ask() loop

In [ ]:
#| export
def _dispatch_tool(name: str, args: dict, root: Path) -> str:
    "Execute a tool call and return the JSON-serialized result."
    if name == 'get_summaries':
        result = _read_summaries(args['video_id'], root)
    elif name == 'get_xscript_range':
        result = _read_xscript_range(args['video_id'], args['start'], args['end'], root)
    else:
        result = {'error': f'Unknown tool: {name}'}
    return json.dumps(result, ensure_ascii=False)

In [ ]:
# Test: _dispatch_tool routes to correct functions
from tempfile import TemporaryDirectory

with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID_DT'; v.mkdir()
    (v / 'summaries.json').write_text(json.dumps({
        'video': {'id': 'VID_DT'}, 'sections': [], 'full': {}}))
    (v / 'captions.en.srt').write_text(
        '1\n00:00:00,000 --> 00:00:03,000\nhello\n')

    r1 = json.loads(_dispatch_tool('get_summaries', {'video_id': 'VID_DT'}, root))
    assert r1['video']['id'] == 'VID_DT'

    r2 = json.loads(_dispatch_tool('get_xscript_range',
                                    {'video_id': 'VID_DT', 'start': 0, 'end': 10}, root))
    assert isinstance(r2, list)

    r3 = json.loads(_dispatch_tool('get_summaries', {'video_id': 'NOPE'}, root))
    assert 'error' in r3

    r4 = json.loads(_dispatch_tool('bad_tool', {}, root))
    assert 'error' in r4
print('ok')

In [ ]:
#| export
import openai

def ask(question: str, # Natural-language query
        video_ids: list[str], # Cached video IDs
        model: str = 'gpt-4o', # LLM model
        max_iterations: int = 20, # Safety cap on tool-use loop
        root: Path = None, # Cache root directory
        verbose: bool = True # Print tool calls to stderr
       ) -> dict: # {answer: str, citations: [{video_id, seconds}]}
    "Run a tool-use loop to answer a question about a video course."
    from yttoc.fetch import _DEFAULT_ROOT
    root = root or _DEFAULT_ROOT
    client = openai.OpenAI()

    user_input = f"Video IDs: {json.dumps(video_ids)}\n---\nQuestion: {question}"

    response = client.responses.create(
        model=model,
        instructions=SYSTEM_PROMPT,
        input=user_input,
        tools=TOOLS,
        text={"format": ANSWER_SCHEMA},
    )

    for _ in range(max_iterations):
        tool_calls = [item for item in response.output
                      if item.type == 'function_call']
        if not tool_calls:
            break

        tool_outputs = []
        for tc in tool_calls:
            args = json.loads(tc.arguments)
            if verbose:
                print(f'{tc.name}({", ".join(f"{k}={v!r}" for k,v in args.items())})',
                      file=sys.stderr)
            result = _dispatch_tool(tc.name, args, root)
            tool_outputs.append({
                'type': 'function_call_output',
                'call_id': tc.call_id,
                'output': result,
            })

        response = client.responses.create(
            model=model,
            previous_response_id=response.id,
            input=tool_outputs,
        )

    text_items = [item for item in response.output if item.type == 'text']
    if not text_items:
        return {'answer': 'No answer generated.', 'citations': []}
    return json.loads(text_items[0].text)

In [ ]:
#| eval: false
# Integration test: ask() with a real cached video (requires OPENAI_API_KEY)
from yttoc.fetch import _DEFAULT_ROOT

result = ask('What is this course about?', ['7T83srD0Mu4'], root=_DEFAULT_ROOT)
print(f"Answer: {result['answer'][:200]}")
print(f"Citations: {result['citations']}")
assert 'answer' in result
assert isinstance(result['citations'], list)